# TMLR AI Reviewer Evaluation
Evaluates AI reviewer outputs against ground truth accept/reject criteria.

In [ ]:
import re
from pathlib import Path
import pandas as pd

# Place the extracted test_cases_all/ directory next to this notebook.
BASE_DIR = Path("test_cases_all")

# gt_claims / gt_audience: "Yes", "No", or None (= metric not computed for that criterion).
# pdf_subdir: subfolder used as the paper list when no ground_truths/ folder exists.
# exclusion_key: key into prompt_examples_exclusion.json (papers used as prompt calibration examples).
CATEGORIES = [
    {"name": "accepted",         "path": "accepted",         "gt_claims": "Yes", "gt_audience": "Yes", "pdf_subdir": None,   "exclusion_key": "tmlr_test_cases_final_accepted"},
    {"name": "claims_only_no",   "path": "claims_only_no",   "gt_claims": "No",  "gt_audience": "Yes", "pdf_subdir": None,   "exclusion_key": "tmlr_test_cases_final_claims_only_no"},
    {"name": "both_no",          "path": "both_no",          "gt_claims": "No",  "gt_audience": "No",  "pdf_subdir": None,   "exclusion_key": "tmlr_test_cases_final_both_no"},
    {"name": "flaws_gemini",     "path": "flaws_gemini",     "gt_claims": "No",  "gt_audience": None,  "pdf_subdir": "pdfs", "exclusion_key": None},
    {"name": "flaws_openai",     "path": "flaws_openai",     "gt_claims": "No",  "gt_audience": None,  "pdf_subdir": "pdfs", "exclusion_key": None}
]

# Map output folder name → human-readable agent display name.
AGENT_NAMES = {
    "output_cspaper":     "CSPaper",
    "output_opus47":      "Opus_4.7",
    "output_gpt55_v3":    "GPT5.5_v3",
    "output_reviewertoo": "ReviewerToo",
}

CLAIMS_QUESTION   = "are the claims made in the submission supported by accurate"
AUDIENCE_QUESTION = "would at least some individuals in tmlr"

In [ ]:
def parse_output(path: Path) -> dict:
    """Extract Claims and Audience Yes/No answers from an AI reviewer .md file."""
    text = path.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines()

    # Strip YAML frontmatter (first --- ... --- block)
    start = 0
    if lines and lines[0].strip() == "---":
        for i, line in enumerate(lines[1:], 1):
            if line.strip() == "---":
                start = i + 1
                break
    body = lines[start:]

    claims_ans = "MISSING"
    audience_ans = "MISSING"
    target = None

    for line in body:
        stripped = line.strip()
        lower = stripped.lower()

        if CLAIMS_QUESTION in lower:
            target = "claims"
            continue
        if AUDIENCE_QUESTION in lower:
            target = "audience"
            continue

        if target and stripped:
            # Strip markdown bold/italic markers (e.g. **Yes**, *No*)
            clean = re.sub(r'\*+', '', stripped).strip()
            answer = clean.capitalize()
            if answer in ("Yes", "No"):
                if target == "claims" and claims_ans == "MISSING":
                    claims_ans = answer
                elif target == "audience" and audience_ans == "MISSING":
                    audience_ans = answer
            target = None

    return {"claims": claims_ans, "audience": audience_ans}

In [ ]:
def ai_decision(claims: str, audience: str) -> str:
    if "MISSING" in (claims, audience):
        return "MISSING"
    return "Accept" if claims == "Yes" and audience == "Yes" else "Reject"


def scan_category(cat: dict) -> list[dict]:
    """
    Scan one category and return per-paper rows.
    - gt_claims / gt_audience may be None (= any answer accepted, metric set to NaN).
    - If pdf_subdir is set, paper list comes from that PDF folder instead of ground_truths/.
    - Agent names are resolved via AGENT_NAMES (keyed by output folder name).
    - Only folders whose name appears in AGENT_NAMES are included; others are skipped.
    - Papers without AI output appear with NaN accuracy columns (pending=True).
    - Supports two filename conventions:
        * {paper_title}__{agent_id}.md  (CSPaper style)
        * {paper_title}.txt             (ReviewerToo metareview style — agent from folder name)
    """
    gt_claims   = cat["gt_claims"]   # "Yes", "No", or None
    gt_audience = cat["gt_audience"] # "Yes", "No", or None
    pdf_subdir  = cat["pdf_subdir"]
    cat_dir     = BASE_DIR / cat["path"]
    cat_name    = cat["name"]

    # GT decision is deterministic whenever at least one criterion is a hard "No" or both are "Yes".
    if gt_claims == "No" or gt_audience == "No":
        gt_dec = "Reject"
    elif gt_claims == "Yes" and gt_audience == "Yes":
        gt_dec = "Accept"
    else:
        gt_dec = "Unknown"

    # Get the canonical paper list
    if pdf_subdir is None:
        papers = {re.sub(r'_AE\.txt$', '', f.name)
                  for f in (cat_dir / "ground_truths").glob("*.txt")}
    else:
        papers = {re.sub(r'\.pdf$', '', f.name)
                  for f in (cat_dir / pdf_subdir).glob("*.pdf")}

    # Collect AI outputs from output*/ subdirs listed in AGENT_NAMES only.
    ai_results: dict[tuple, dict] = {}
    for output_dir in sorted(cat_dir.glob("output*/")):
        if not output_dir.is_dir():
            continue
        if output_dir.name not in AGENT_NAMES:
            continue
        agent_name = AGENT_NAMES[output_dir.name]
        for review_file in output_dir.glob("*"):
            if review_file.suffix not in (".md", ".txt"):
                continue
            stem = review_file.stem
            if "__" in stem:
                paper_title = stem.rsplit("__", 1)[0]
            else:
                paper_title = stem
            ai_results[(paper_title, agent_name)] = parse_output(review_file)

    agents = sorted({ag for (_, ag) in ai_results}) or [None]

    rows = []
    for pt in sorted(papers):
        for ag in agents:
            result = ai_results.get((pt, ag))
            if result is None:
                rows.append(dict(
                    paper=pt, category=cat_name, agent_id=ag,
                    gt_claims=gt_claims, gt_audience=gt_audience, gt_decision=gt_dec,
                    ai_claims=float("nan"), ai_audience=float("nan"), ai_decision=float("nan"),
                    claims_correct=float("nan"), audience_correct=float("nan"),
                    both_correct=float("nan"), decision_correct=float("nan"),
                    pending=True,
                ))
            else:
                ai_c, ai_a = result["claims"], result["audience"]
                ai_dec = ai_decision(ai_c, ai_a)

                cc = (ai_c == gt_claims)   if gt_claims   is not None else float("nan")
                ac = (ai_a == gt_audience) if gt_audience is not None else float("nan")
                bc = (bool(cc) and bool(ac)) if (gt_claims is not None and gt_audience is not None) else float("nan")
                dc = (ai_dec == gt_dec) if (ai_dec != "MISSING" and gt_dec != "Unknown") else float("nan")

                rows.append(dict(
                    paper=pt, category=cat_name, agent_id=ag,
                    gt_claims=gt_claims, gt_audience=gt_audience, gt_decision=gt_dec,
                    ai_claims=ai_c, ai_audience=ai_a, ai_decision=ai_dec,
                    claims_correct=cc, audience_correct=ac,
                    both_correct=bc, decision_correct=dc,
                    pending=False,
                ))
    return rows

In [ ]:
# Build the full DataFrame
all_rows = []
for cat in CATEGORIES:
    all_rows.extend(scan_category(cat))

df = pd.DataFrame(all_rows)
print(f"Total rows: {len(df)}")
print(f"Categories: {df['category'].unique()}")
print(f"Agents found: {df['agent_id'].dropna().unique()}")
df.head()

In [ ]:
# Per-category response distribution — ELIGIBLE + COMMON PAPERS
# Eligible = not used as a training example in either prompt (excluded_pairs).
# Common  = reviewed by ALL THREE agents (CSPaper, GPT5.5_v3, Opus_4.7).
# → N is identical across agents within each category.

import json

def fmt(v):
    return f"{v:.1f}%" if pd.notna(v) else "N/A"

FOCUS_AGENTS_3 = {"CSPaper", "GPT5.5_v3", "Opus_4.7"}

# Union of both exclusion lists
e1 = json.loads(Path("claude_evals/prompt_examples_exclusion.json").read_text())["excluded_papers"]
e2 = json.loads(Path("openai_evals/prompt_examples_exclusion.json").read_text())["excluded_papers"]
excl_union = {k: set(e1.get(k, [])) | set(e2.get(k, [])) for k in set(e1) | set(e2)}

cat_excl_key = {c["name"]: c.get("exclusion_key") for c in CATEGORIES}

excluded_pairs = {
    (cat_name, paper)
    for cat_name, key in cat_excl_key.items() if key
    for paper in excl_union.get(key, set())
}

# Start from non-pending rows for the 3 agents
focus_df = df[df["agent_id"].isin(FOCUS_AGENTS_3) & ~df["pending"]].copy()
focus_df["_pair"] = list(zip(focus_df["category"], focus_df["paper"]))
eligible_df = focus_df[~focus_df["_pair"].isin(excluded_pairs)].drop(columns="_pair")

# Keep only (category, paper) pairs reviewed by all 3 agents
coverage = eligible_df.groupby(["category", "paper"])["agent_id"].nunique()
common_keys = coverage[coverage == len(FOCUS_AGENTS_3)].reset_index()[["category", "paper"]]
common_df = eligible_df.merge(common_keys, on=["category", "paper"])

# Distribution table (claims only)
elig_rows = []
for (cat, agent), grp in common_df.groupby(["category", "agent_id"], sort=False):
    gt_c = grp["gt_claims"].iloc[0]
    elig_rows.append({
        "category":      cat,
        "agent":         agent,
        "N":             len(grp),
        "claims_GT":     gt_c if gt_c is not None else "—",
        "claims→Yes%":   (grp["ai_claims"] == "Yes").mean() * 100,
        "claims→No%":    (grp["ai_claims"] == "No").mean()  * 100,
    })

elig_summary_df = pd.DataFrame(elig_rows)
print("=== Per-Category Claims Distribution (Eligible + Common Papers) ===")
elig_summary_df.style.format({
    "claims→Yes%": fmt,
    "claims→No%":  fmt,
}).hide(axis="index")